# 用 MultipleNegativesRankingLoss 訓練句向量模型

用 MNLI 的「蘊含」句對組成 `(anchor, positive, negative)` 三元組，以 `MultipleNegativesRankingLoss` 微調 `bert-base-uncased`，最後用 STS-B 驗證集評估句子相似度。

```
MNLI 前 50000 筆 → 只留 label == 0（蘊含）
   → premise = anchor, hypothesis = positive, 打亂的 hypothesis = negative
   → bert-base-uncased + MNR Loss 訓練 1 epoch
   → STS-B 驗證集評估 (cosine)
```

## 0. Import

已移除原本多餘的 `sympy`、`torch.embedding`、`trainer` 三個 import。

In [ ]:
import random

from datasets import load_dataset, Dataset
from tqdm import tqdm

from sentence_transformers import SentenceTransformer
from sentence_transformers.sentence_transformer import losses
from sentence_transformers.sentence_transformer.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers.sentence_transformer.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.trainer import SentenceTransformerTrainer

## 1. 建立三元組資料集

- MNLI 標籤：`0 = 蘊含`、`1 = 中立`、`2 = 矛盾`
- `filter(lambda x: x['label'] == 0)`：只保留蘊含，因為 premise 和 hypothesis 語意相近，適合當正例
- `list(...)`：新版 `datasets` 取出的欄位是 `Column` 物件，不能直接 `random.shuffle`，要先轉成 list
- 把 hypothesis 打亂，每個 premise 會配到別人的 hypothesis，當作 **soft negative**

| 欄位 | 來源 | 角色 |
|---|---|---|
| `anchor` | `premise` | 基準句 |
| `positive` | 配對的 `hypothesis` | 應該靠近 anchor |
| `negative` | 隨機別人的 `hypothesis` | 應該遠離 anchor |

In [ ]:
def load_dataset_triplet():
    mnli_dataset = load_dataset("nyu-mll/glue", "mnli", split="train").select(range(50000))
    mnli_dataset = mnli_dataset.remove_columns("idx")
    mnli_dataset = mnli_dataset.filter(lambda x: x["label"] == 0)
    print("蘊含筆數:", len(mnli_dataset))
    print(mnli_dataset[0])

    # 打亂 hypothesis 當作 soft negative
    soft_negatives = list(mnli_dataset["hypothesis"])
    random.shuffle(soft_negatives)

    triplets = {"anchor": [], "positive": [], "negative": []}
    for row, neg in tqdm(zip(mnli_dataset, soft_negatives)):
        # 避免 negative 剛好抽到自己的 positive
        if neg == row["hypothesis"]:
            continue
        triplets["anchor"].append(row["premise"])
        triplets["positive"].append(row["hypothesis"])
        triplets["negative"].append(neg)

    return Dataset.from_dict(triplets)


train_dataset = load_dataset_triplet()
print(train_dataset)
print(train_dataset[0])

### `for row, neg in zip(...)` 是什麼

`zip` 把兩個序列並排，每一圈各拿一個：

| 變數 | 來自 | 型別 | 內容 |
|---|---|---|---|
| `row` | `mnli_dataset` | dict | `{'premise': ..., 'hypothesis': ..., 'label': 0}` |
| `neg` | 打亂後的清單 | str | 某個隨機的 hypothesis |

## 2. 建立評估器（STS-B）

- STS-B 人工分數範圍 0~5，除以 5 正規化成 0~1
- `main_similarity="cosine"`：算兩句向量的餘弦相似度，再和人工分數算相關係數
- STS-B 沒有被拿來訓練，用來檢查泛化能力

In [ ]:
def load_evaluator():
    val_sts = load_dataset("nyu-mll/glue", "stsb", split="validation")
    return EmbeddingSimilarityEvaluator(
        sentences1=val_sts["sentence1"],
        sentences2=val_sts["sentence2"],
        scores=[score / 5 for score in val_sts["label"]],
        main_similarity="cosine",
    )


stsb_evaluator = load_evaluator()

## 3. 模型、Loss、訓練參數

### MultipleNegativesRankingLoss 原理

一個 batch 裡，每個 anchor 只有一個正確答案（自己的 positive），**其他所有句子都當負例**。

batch size = 32 時，對第 i 個 anchor：

- 正例：第 i 個 positive
- 負例：其他 31 個 positive + 全部 32 個 negative

做法：算 anchor 對所有 positive 與 negative 的餘弦相似度（32 × 64 矩陣），乘上 scale（預設 20），每列做 cross entropy。

重點：
- 不需要 label 欄位，只要欄位順序是 `anchor, positive, negative`
- batch size 越大，負例越多，通常效果越好
- batch 內若有重複句子，會被當成負例（假負例）

### 訓練參數

| 參數 | 值 | 說明 |
|---|---|---|
| `output_dir` | `mnr_embedding_model` | 輸出資料夾（checkpoint、記錄），不等於最終模型 |
| `num_train_epochs` | 1 | 資料只看一遍 |
| `per_device_train_batch_size` | 32 | 每步 32 筆，同時決定 MNR 的負例數量 |
| `warmup_steps` | 100 | 學習率前 100 步從 0 升到預設的 `5e-5` |
| `fp16` | True | 混合精度，需要支援的 GPU |
| `eval_strategy` | `"steps"` | **要加這個，`eval_steps` 才會生效** |
| `eval_steps` / `logging_steps` | 100 | 每 100 步評估 / 記錄 |

In [ ]:
embedding_model = SentenceTransformer("bert-base-uncased")

mnr_loss = losses.MultipleNegativesRankingLoss(model=embedding_model)

args = SentenceTransformerTrainingArguments(
    output_dir="mnr_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_strategy="steps",   # 沒設這個，eval_steps 不會生效
    eval_steps=100,
    logging_steps=100,
    # 避免同一 batch 出現重複句子造成假負例（import 路徑依版本而定，需要時再啟用）
    # batch_sampler=BatchSamplers.NO_DUPLICATES,
)

## 4. 建立 Trainer

`SentenceTransformerTrainer` 把模型、資料、loss、評估器組在一起。

In [ ]:
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=mnr_loss,
    evaluator=stsb_evaluator,
)

## 5. 訓練前後對比

`trainer.train()` 會**原地修改** `embedding_model`，所以訓練後再評估一次，就是訓練後的分數。

In [ ]:
print("before:", stsb_evaluator(embedding_model))

In [ ]:
trainer.train()

In [ ]:
print("after:", stsb_evaluator(embedding_model))

## 6. 儲存最終模型

`output_dir` 只有 checkpoint，不保證根目錄能直接載入。要保存最終模型，要自己呼叫 `save`。

In [ ]:
embedding_model.save("mnr_final_model")

之後要重新評估、不用重新訓練：

In [ ]:
loaded_model = SentenceTransformer("mnr_final_model")
print(stsb_evaluator(loaded_model))

## 7. 互動環境清顯存（選用）

在 Jupyter 裡，行程會一直活著，重複訓練前建議清掉舊物件，或直接 Restart Kernel。terminal 執行 `python test.py` 則不需要，程式結束會自動釋放。

In [ ]:
import gc
import torch

del trainer, embedding_model, mnr_loss
gc.collect()
torch.cuda.empty_cache()

## 重點整理

1. **目標**：用 NLI 蘊含句對訓練句向量，用 STS-B 檢查相似度表現。
2. **資料**：MNLI 只留 `label == 0`，組成 `(anchor, positive, negative)`。
3. **negative**：把 hypothesis 打亂，配到別人身上，屬於隨機（soft）負例。
4. **loss**：MNR Loss 把 batch 內其他樣本都當負例，不需要 label，batch 越大越好。
5. **評估**：STS-B 分數除以 5，用 cosine 相似度算相關係數。
6. **常見坑**：`Column` 不能 shuffle（要 `list()`）、`eval_steps` 要配 `eval_strategy`、`output_dir` 不是最終模型、變數同名、多餘 import。
7. **離線環境**：模型和資料集已快取時，可設 `HF_HUB_OFFLINE=1` 避免一直重試連線。